# ⏳ Day 04a: FastAPI Background Tasks

---

## 🎯 What You'll Master Today
- FastAPI BackgroundTasks — when and how to use
- BackgroundTasks vs Celery vs async/await
- Common patterns: email, logging, file processing
- Limitations and best practices

---

## 🎭 The Receptionist Analogy

A receptionist takes your name and says "We'll send your confirmation email." They don't make you wait at the desk while they compose and send it — they respond immediately and do the email **after**.

That's BackgroundTasks: the endpoint returns a response first, then runs the background work in the same process.

```
╔═══════════════════════════════════════════════════════════════════╗
║  BackgroundTasks vs Celery vs async/await                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  BackgroundTasks:                                                ║
║    ✅ Simple, no extra infrastructure                            ║
║    ✅ Fire-and-forget after response                             ║
║    ❌ No retry, no monitoring, no result tracking                ║
║    ❌ Lost if server crashes                                     ║
║    Best for: logging, simple emails, cache updates              ║
║                                                                   ║
║  Celery:                                                         ║
║    ✅ Retry, monitoring, scheduling, distributed                 ║
║    ❌ Needs Redis/RabbitMQ infrastructure                        ║
║    Best for: heavy tasks, payment processing, reports           ║
║                                                                   ║
║  async/await:                                                    ║
║    ✅ Concurrent I/O within the request                          ║
║    ❌ Blocks response until complete                             ║
║    Best for: parallel API calls during request                  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Basic BackgroundTasks
# ============================================================
from fastapi import FastAPI, BackgroundTasks
from fastapi.testclient import TestClient
import time

app = FastAPI()

# Background function (regular function, not async)
def write_log(message: str):
    time.sleep(0.5)  # Simulate slow I/O
    with open("app.log", "a") as f:
        f.write(f"{message}\n")
    print(f"  [BG] Log written: {message}")

def send_notification(email: str, message: str):
    time.sleep(1)  # Simulate email sending
    print(f"  [BG] Email sent to {email}: {message}")

@app.post("/users/")
def create_user(username: str, email: str, background_tasks: BackgroundTasks):
    # Do the main work
    user = {"username": username, "email": email, "id": 1}
    
    # Schedule background work AFTER response is sent
    background_tasks.add_task(write_log, f"User created: {username}")
    background_tasks.add_task(send_notification, email, "Welcome!")
    
    return user  # Client gets this immediately!

client = TestClient(app)
start = time.perf_counter()
r = client.post("/users/?username=sarthak&email=s@m.com")
elapsed = time.perf_counter() - start
print(f"\nResponse: {r.json()}")
print(f"Response time: {elapsed:.3f}s (BG tasks run after response)")

In [ ]:
# ============================================================
# BackgroundTasks in Dependencies
# ============================================================
from fastapi import Depends

app2 = FastAPI()

def log_request(background_tasks: BackgroundTasks):
    """Dependency that adds logging as background task."""
    # This runs BEFORE the endpoint
    # But the background task runs AFTER the response
    background_tasks.add_task(
        write_log, f"Request at {time.time():.0f}"
    )

@app2.get("/items/", dependencies=[Depends(log_request)])
def get_items(background_tasks: BackgroundTasks):
    # Can add more BG tasks in the endpoint too
    background_tasks.add_task(write_log, "Items fetched")
    return [{"item": "Widget"}]

client2 = TestClient(app2)
r = client2.get("/items/")
print(f"Items: {r.json()}")
print("✅ BG tasks from both dependency and endpoint run after response")

In [ ]:
# ============================================================
# Async Background Tasks
# ============================================================

app3 = FastAPI()

# Background tasks can be async too!
async def async_notification(user_id: int, action: str):
    import asyncio
    await asyncio.sleep(0.5)  # Simulate async I/O
    print(f"  [BG-async] User {user_id}: {action}")

@app3.post("/orders/")
async def create_order(background_tasks: BackgroundTasks):
    order = {"id": 42, "status": "created"}
    
    # async background task
    background_tasks.add_task(async_notification, 1, "order_created")
    
    return order

client3 = TestClient(app3)
r = client3.post("/orders/")
print(f"Order: {r.json()}")
print("\n💡 FastAPI handles both sync and async BG tasks automatically")

In [ ]:
# ============================================================
# Real-World Patterns
# ============================================================

print("""
# Pattern 1: File Processing
def process_uploaded_file(file_path: str, user_id: int):
    # Resize image, extract text, etc.
    result = heavy_processing(file_path)
    update_db(user_id, result)

@app.post("/upload/")
async def upload(file: UploadFile, background_tasks: BackgroundTasks):
    path = f"/tmp/{file.filename}"
    with open(path, "wb") as f:
        f.write(await file.read())
    background_tasks.add_task(process_uploaded_file, path, current_user.id)
    return {"status": "processing"}


# Pattern 2: Cache Warming
def warm_cache(key: str, data: dict):
    redis.setex(key, 3600, json.dumps(data))

@app.get("/dashboard/")
async def dashboard(background_tasks: BackgroundTasks):
    data = fetch_dashboard_data()
    background_tasks.add_task(warm_cache, "dashboard:v1", data)
    return data


# Pattern 3: Audit Logging
def audit_log(user_id: int, action: str, details: dict):
    db.execute(
        "INSERT INTO audit_log (user_id, action, details, ts) VALUES ...",
        {"user_id": user_id, "action": action, "details": json.dumps(details)}
    )

@app.delete("/users/{user_id}")
async def delete_user(user_id: int, background_tasks: BackgroundTasks):
    db.delete_user(user_id)
    background_tasks.add_task(audit_log, user_id, "deleted", {"reason": "request"})
    return {"deleted": True}
""")

print("Good use cases for BackgroundTasks:")
print("  ✅ Logging / audit trails")
print("  ✅ Cache warming / invalidation")
print("  ✅ Simple notifications / emails")
print("  ✅ Post-response cleanup")
print("")
print("Use Celery instead when:")
print("  ❌ Task takes > 30 seconds")
print("  ❌ Need retry logic")
print("  ❌ Need to check status later")
print("  ❌ Task must survive server restart")

---

## 🗺️ Background Processing Decision Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  WHICH BACKGROUND APPROACH?                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Question: Does it need to complete during this request?        ║
║    YES → async/await (gather concurrent I/O)                    ║
║    NO  → Continue below                                         ║
║                                                                   ║
║  Question: Is the task < 5 seconds? No retry needed?            ║
║    YES → BackgroundTasks (simple, in-process)                   ║
║    NO  → Continue below                                         ║
║                                                                   ║
║  Question: Need retry, monitoring, or scheduling?               ║
║    YES → Celery + Redis (distributed, reliable)                 ║
║    NO  → BackgroundTasks is fine                                ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is BackgroundTasks? | FastAPI built-in for running code after response is sent. In-process, no broker |
| 2 | BackgroundTasks vs Celery? | BG: simple, no infra, no retry. Celery: distributed, retry, monitoring |
| 3 | When would BG task fail? | Server crash during execution — task lost. No retry mechanism |
| 4 | Can BG tasks be async? | Yes! add_task() accepts both sync and async functions |
| 5 | How to add BG tasks from deps? | Inject BackgroundTasks into dependency, add_task() same way |
| 6 | BackgroundTasks vs asyncio.create_task? | BG: runs after response. create_task: runs during request |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • BackgroundTasks runs code AFTER response              │
## │  • add_task(func, arg1, arg2) — simple API              │
## │  • Works with both sync and async functions              │
## │  • Use for: logging, email, cache, light cleanup        │
## │  • No retry, no monitoring — use Celery for heavy work  │
## │  • Can inject BackgroundTasks in dependencies too       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 06 — Deployment with Docker, Compose & CI/CD**